Zelle 1: Setup & Imports

## Job Description Classification with SetFit and OOD Detection

This notebook demonstrates an approach to classifying job descriptions while identifying Out-of-Distribution (OOD) samples. We leverage **SetFit**, a powerful few-shot fine-tuning framework for Sentence-Transformers, to train a highly accurate text classification model on clean, in-distribution job titles.

Crucially, the training data is cleaned to exclude any 'Other' category, ensuring the model learns only the characteristics of known job roles and does not treat Other as a regular class. This is later important for For OOD detection, we can flag job descriptions that do not conform to any of the learned categories, providing a mechanism for handling novel or ambiguous input.

In [ ]:
!pip install transformers torch scikit-learn scipy pandas datasets matplotlib seaborn

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel
from sklearn.covariance import EmpiricalCovariance
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

# GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


Data Loading & Clean Split 
Training (ID all 10 classes) vs. OOD ("Other")

In [ ]:
# GPU Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Load data
!wget -O department-v2.csv https://raw.githubusercontent.com/konradfuniwue/PracticalDataScience/refs/heads/main/Data/Original%20Data/department-v2.csv

df_train_raw = pd.read_csv('department-v2.csv').rename(columns={'job_description': 'text'})


# 2. Define "Other"
OOD_LABEL = 'Other'

# 3. Clean TRAINING SET (Learn only In-Distribution!)
# We remove all "Other" rows from the training data.
df_train_id = df_train_raw[df_train_raw['label'] != OOD_LABEL].copy()

# 4. Train-Test Split for ID-Data
# Here we insert the Train-Test Split
# df_train_id now contains the ID data WITHOUT 'Other'

# Splitting the In-Distribution data into training and test sets
# We use 'stratify' to ensure the class distribution is the same in both sets.
df_train_id, df_test_id, _, _ = train_test_split(
    df_train_id, df_train_id['label'], test_size=0.2, random_state=42, stratify=df_train_id['label']
)

# OOD-Test: The "Other" cases (to check OOD detection)
# We take 'Other' from Test AND Training (which we removed above) as OOD examples
ood_from_train = df_train_raw[df_train_raw['label'] == OOD_LABEL]
df_test_ood =  ood_from_train

print(f"Training (Clean ID): {len(df_train_id)}")
print(f"Test (Clean ID):     {len(df_test_id)}")
print(f"Test (OOD/Other):    {len(df_test_ood)}")

Using device: cuda
--2026-01-30 22:58:40--  https://raw.githubusercontent.com/konradfuniwue/PracticalDataScience/refs/heads/main/department-v2.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 468519 (458K) [text/plain]
Saving to: ‘department-v2.csv’

department-v2.csv   100%[===================>] 457.54K  --.-KB/s    in 0.02s   

2026-01-30 22:58:40 (22.0 MB/s) - ‘department-v2.csv’ saved [468519/468519]

--2026-01-30 22:58:40--  https://raw.githubusercontent.com/konradfuniwue/PracticalDataScience/main/linkedin-cvs-annotated.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP

In [ ]:
!pip install setfit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00


sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2, SetFitModel (fine tuning)


In [ ]:
from setfit import SetFitModel, SetFitTrainer
from sentence_transformers.losses import CosineSimilarityLoss
from datasets import Dataset

# 1. Prepare Dataset
train_dataset = Dataset.from_pandas(df_train_id)

# 2. Initialize Model
unique_labels = df_train_id['label'].unique().tolist()
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    labels=unique_labels
)

# 3. Create Trainer
trainer = SetFitTrainer(
    model=model,
    train_dataset=train_dataset,
    loss_class=CosineSimilarityLoss,
    batch_size=16,
    num_iterations=20,
    num_epochs=1
)

# 4. Train
print("Starting Fine-Tuning...")
trainer.train()

# 5. Save Body
model.model_body.save_pretrained("fine_tuned_job_embeddings")
print("Embedding model saved.")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/wandb/analytics/sentry.py:268: DeprecationWarning: Read the `app_url` setting from the appropriate Settings object.
  app_url = wandb.util.app_url(tags["base_url"])  # type: ignore[index]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)


Step,Training Loss
1,0.143900
50,0.209700
100,0.204800
150,0.208900
200,0.177900
250,0.140900
300,0.108700
350,0.078700
400,0.066300
450,0.052700


/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.pin_memory() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:46.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/google/protobuf/internal/well_known_types.py:178: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  self.FromDatetime(datetime.datetime.utcnow())
/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py:57: DeprecationWarning: The argument 'device' of Tensor.is_pinned() is deprecated. Please do not pass this argument. (Triggered internally at /pytorch/aten/src/ATen/native/Memory.cpp:31.)
  return data.pin_memory(device)
/usr/local/lib/python3.12/dist-packages/jupyter_client/sessi

Embedding-Modell gespeichert.


In [ ]:
import shutil
from google.colab import files

# Path to directory
model_dir = "fine_tuned_job_embeddings"
zip_file_name = "fine_tuned_job_embeddings.zip"

# zip the archive
shutil.make_archive(model_dir, 'zip', model_dir)

print(f"'{zip_file_name}' got created")

#Open Download
files.download(zip_file_name)

'fine_tuned_job_embeddings.zip' wurde erstellt. Sie können es jetzt herunterladen.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>